# COCO → SAM2 → U-Net Mask Converter

Pipeline:
1. Read COCO JSON (bbox annotations).
2. Use SAM2 with bbox prompts to generate per-instance masks.
3. Compose multi-class semantic segmentation masks (0=bg, 1..N=classes).
4. Save `images/` + `masks/` pairs for U-Net training.

Mask encoding: single-channel PNG, pixel value = class id (0=bg). Use `class_id + 1` so background stays 0.

Dataset: `dataset_augmented_04_23_2026/dataset_augmented/{train,val,test}` with `_annotations.coco.json` + `images/`.
Categories: Person(0), Car(1), OtherVehicle(2).

## 1. Install deps

Run once. SAM2 from Meta repo. Needs CUDA-capable GPU for speed (CPU works but slow).

In [ ]:
# !pip install -q torch torchvision opencv-python pillow numpy tqdm pycocotools
# !pip install -q git+https://github.com/facebookresearch/sam2.git
# Download SAM2 checkpoint (pick one):
#   sam2.1_hiera_tiny.pt | sam2.1_hiera_small.pt | sam2.1_hiera_base_plus.pt | sam2.1_hiera_large.pt
# !mkdir -p checkpoints && wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt

## 2. Imports & config

In [ ]:
import json, os, shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import cv2
from PIL import Image
import torch
from tqdm.auto import tqdm

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# ---- paths ----
ROOT       = Path('dataset/dataset_augmented_04_23_2026/dataset_augmented')
OUT_ROOT   = Path('dataset/dataset_unet_sam2')
SPLITS     = ['train', 'val', 'test']

SAM2_CFG   = 'configs/sam2.1/sam2.1_hiera_b+.yaml'   # match checkpoint variant
SAM2_CKPT  = 'checkpoints/sam2.1_hiera_base_plus.pt'

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_BF16   = DEVICE == 'cuda'                         # bf16 only on CUDA
BBOX_BATCH = 64                                       # bboxes per SAM2 forward
OVERWRITE  = False                                    # skip existing masks

print('device:', DEVICE)

## 3. Load SAM2

In [ ]:
sam2_model = build_sam2(SAM2_CFG, SAM2_CKPT, device=DEVICE)
predictor  = SAM2ImagePredictor(sam2_model)
print('SAM2 ready')

## 4. COCO index helper

In [ ]:
def load_coco(json_path: Path):
    with open(json_path) as f:
        coco = json.load(f)
    imgs = {im['id']: im for im in coco['images']}
    anns_by_img = defaultdict(list)
    for a in coco['annotations']:
        anns_by_img[a['image_id']].append(a)
    cats = {c['id']: c['name'] for c in coco['categories']}
    return imgs, anns_by_img, cats

# class_id → mask value (bg=0). Keep COCO id+1 so 0 stays background.
def class_to_pixel(cat_id: int) -> int:
    return cat_id + 1

## 5. Per-image conversion

Strategy:
- Set image once on predictor.
- Batch all bboxes for that image through SAM2 (`predict(box=...)`).
- Paint each returned mask onto semantic mask with its class pixel value.
- Smaller objects painted last → they win overlaps (paint by descending bbox area).

In [ ]:
@torch.inference_mode()
def masks_for_image(image_bgr, anns):
    """Return list of (class_pixel, binary_mask, area) sorted big→small."""
    if not anns:
        return []
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    predictor.set_image(image_rgb)

    # COCO bbox = [x,y,w,h] → SAM2 expects [x1,y1,x2,y2]
    boxes_xyxy = []
    cats = []
    areas = []
    for a in anns:
        x, y, w, h = a['bbox']
        if w <= 0 or h <= 0:
            continue
        boxes_xyxy.append([x, y, x + w, y + h])
        cats.append(a['category_id'])
        areas.append(w * h)
    if not boxes_xyxy:
        return []
    boxes_np = np.asarray(boxes_xyxy, dtype=np.float32)

    out = []
    for i0 in range(0, len(boxes_np), BBOX_BATCH):
        chunk = boxes_np[i0:i0 + BBOX_BATCH]
        masks, scores, _ = predictor.predict(
            point_coords=None, point_labels=None,
            box=chunk, multimask_output=False,
        )
        # masks shape: (N,1,H,W) or (N,H,W) depending on version
        if masks.ndim == 4:
            masks = masks[:, 0]
        for j, m in enumerate(masks):
            idx = i0 + j
            out.append((class_to_pixel(cats[idx]), m.astype(bool), areas[idx]))
    out.sort(key=lambda t: -t[2])   # big → small (small painted last = wins)
    return out


def compose_semantic_mask(h, w, instances):
    sem = np.zeros((h, w), dtype=np.uint8)
    for px, m, _ in instances:
        sem[m] = px
    return sem

## 6. Run conversion over splits

In [ ]:
amp_ctx = (
    torch.autocast(device_type='cuda', dtype=torch.bfloat16)
    if USE_BF16 else torch.autocast(device_type='cpu', enabled=False)
)

for split in SPLITS:
    src_dir   = ROOT / split
    img_dir   = src_dir / 'images'
    json_path = src_dir / '_annotations.coco.json'
    if not json_path.exists():
        print(f'skip {split}: no json'); continue

    out_img  = OUT_ROOT / split / 'images'
    out_mask = OUT_ROOT / split / 'masks'
    out_img.mkdir(parents=True, exist_ok=True)
    out_mask.mkdir(parents=True, exist_ok=True)

    imgs, anns_by_img, cats = load_coco(json_path)
    print(f'[{split}] imgs={len(imgs)} cats={cats}')

    with amp_ctx:
        for img_id, info in tqdm(imgs.items(), desc=split):
            fname = info['file_name']
            stem  = Path(fname).stem
            mask_path = out_mask / f'{stem}.png'
            if mask_path.exists() and not OVERWRITE:
                continue

            img_path = img_dir / fname
            image_bgr = cv2.imread(str(img_path))
            if image_bgr is None:
                print('missing:', img_path); continue
            h, w = image_bgr.shape[:2]

            instances = masks_for_image(image_bgr, anns_by_img.get(img_id, []))
            sem = compose_semantic_mask(h, w, instances)

            # copy image (link is faster but cross-fs may fail → fallback copy)
            dst_img = out_img / fname
            if not dst_img.exists():
                try:
                    os.symlink(img_path.resolve(), dst_img)
                except OSError:
                    shutil.copy2(img_path, dst_img)
            Image.fromarray(sem, mode='L').save(mask_path, optimize=True)

print('done')

## 7. Sanity check — visualize a few

In [ ]:
import matplotlib.pyplot as plt

PALETTE = np.array([
    [0,   0,   0  ],   # 0 bg
    [255, 0,   0  ],   # 1 Person
    [0,   255, 0  ],   # 2 Car
    [0,   0,   255],   # 3 OtherVehicle
], dtype=np.uint8)

def colorize(mask):
    return PALETTE[mask]

split = 'train'
imgs = sorted((OUT_ROOT / split / 'masks').glob('*.png'))[:4]
fig, axes = plt.subplots(len(imgs), 3, figsize=(12, 4*len(imgs)))
for r, mp in enumerate(imgs):
    stem = mp.stem
    ip = next((OUT_ROOT/split/'images').glob(f'{stem}.*'))
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    m   = np.array(Image.open(mp))
    over = (0.5*img + 0.5*colorize(m)).astype(np.uint8)
    for ax, x, t in zip(axes[r], [img, colorize(m), over], ['image','mask','overlay']):
        ax.imshow(x); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()

## 8. Minimal U-Net `Dataset` class

Drop-in for PyTorch training. Returns `(image_tensor, mask_long_tensor)` — mask is class indices, ready for `CrossEntropyLoss(num_classes=4)`.

In [ ]:
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

class CocoSam2UNetDataset(Dataset):
    def __init__(self, root: Path, split: str, size=512):
        self.img_dir  = Path(root) / split / 'images'
        self.mask_dir = Path(root) / split / 'masks'
        self.items = sorted(self.mask_dir.glob('*.png'))
        self.size = size

    def __len__(self): return len(self.items)

    def __getitem__(self, i):
        mp = self.items[i]
        ip = next(self.img_dir.glob(f'{mp.stem}.*'))
        img = Image.open(ip).convert('RGB').resize((self.size, self.size), Image.BILINEAR)
        m   = Image.open(mp).resize((self.size, self.size), Image.NEAREST)
        x = TF.to_tensor(img)                       # (3,H,W) float [0,1]
        y = torch.from_numpy(np.array(m)).long()    # (H,W) long [0..C]
        return x, y

# ds = CocoSam2UNetDataset(OUT_ROOT, 'train'); print(len(ds), ds[0][0].shape, ds[0][1].unique())